In [1]:
import openml
import pandas as pd
import numpy as np
import os
import traceback
import time
import gc

In [2]:
def local_load_openml_suite(suite_id, task, suite_name, limit=-1):
    """Redefined to ensure no circular imports and clean metadata access."""
    suite = openml.study.get_suite(suite_id)
    datasets = []
    task_iterator = iter(suite.tasks)
    
    target_count = float('inf') if limit == -1 else limit
    
    while len(datasets) < target_count:
        try:
            task_id = next(task_iterator)
            task_obj = openml.tasks.get_task(task_id)
            dataset = task_obj.get_dataset()

            # We still need to pull the data to verify the exact shape used in training
            X, y, _, _ = dataset.get_data(
                target=dataset.default_target_attribute,
                dataset_format="dataframe"
            )

            train_idx, _ = task_obj.get_train_test_split_indices(fold=0)

            datasets.append({
                "name": dataset.name,
                "task": task,
                "data_type": "tabular", 
                "X_train": X.iloc[train_idx],
            })
            print(f"Loaded {suite_name}: {dataset.name}")

        except StopIteration:
            break
        except Exception as e:
            continue

    return datasets

def local_load_aeon_suite(suite_type, limit=-1):
    """Redefined AEON loader for TSC or TSER."""
    from aeon.datasets import load_classification, load_regression, tsc_datasets, tser_datasets
    
    if suite_type == "TSC":
        names = iter(tsc_datasets.univariate_equal_length + tsc_datasets.multivariate_equal_length)
        load_func = load_classification
        task = "classification"
    else:
        names = iter(list(tser_datasets.tser_monash.keys()) + list(tser_datasets.tser_soton_clean))
        load_func = load_regression
        task = "regression"

    datasets = []
    target_count = float('inf') if limit == -1 else limit

    while len(datasets) < target_count:
        try:
            name = next(names)
            X_train, _ = load_func(name, split="train")
            datasets.append({
                "name": name,
                "task": task,
                "data_type": "ts", 
                "X_train": X_train,
            })
            print(f"Loaded {suite_type}: {name}")
        except StopIteration: break
        except Exception: continue
    return datasets

# --- Main Calculation Logic ---

def calculate_benchmark_cells():
    benchmarks = {
        "OpenML-CC18": lambda: local_load_openml_suite(99, "classification", "OpenML-CC18"),
        "OpenML-297": lambda: local_load_openml_suite(297, "regression", "OpenML-297"),
        "AEON-TSC": lambda: local_load_aeon_suite("TSC"),
        "AEON-TSER": lambda: local_load_aeon_suite("TSER")
    }

    summary_results = []
    dataset_details = []

    print("Starting Standalone EDA: Training Cell Count Analysis...\n")

    for benchmark_name, loader_func in benchmarks.items():
        print(f"--- Processing {benchmark_name} ---")
        datasets = loader_func()
        
        benchmark_total_cells = 0
        valid_count = 0

        for ds in datasets:
            try:
                raw_X = ds["X_train"]
                
                # Fast shape inspection
                if hasattr(raw_X, "shape"):
                    shape = raw_X.shape
                    
                    if len(shape) == 3: # Time Series (N, C, L)
                        rows, cols = shape[0], shape[1] * shape[2]
                    elif len(shape) == 2: # Tabular
                        rows, cols = shape
                    else: # 1D or other
                        rows, cols = shape[0], 1
                    
                    cells = rows * cols
                    benchmark_total_cells += cells
                    valid_count += 1
                    
                    dataset_details.append({
                        "Benchmark": benchmark_name,
                        "Dataset": ds["name"],
                        "Rows": rows,
                        "Cols": cols,
                        "Cells": cells
                    })

                # Aggressive Memory Cleanup
                ds["X_train"] = None
                del raw_X

            except Exception as e:
                print(f"Error on {ds.get('name')}: {e}")

        gc.collect() # Clear memory before starting the next benchmark suite

        summary_results.append({
            "Benchmark": benchmark_name,
            "Datasets": valid_count,
            "Total Cells": benchmark_total_cells,
            "Avg Cells": round(benchmark_total_cells / valid_count) if valid_count > 0 else 0
        })
        print(f"Subtotal for {benchmark_name}: {benchmark_total_cells:,}\n")

    # Output
    df_summary = pd.DataFrame(summary_results)
    print("\n" + "="*30 + "\nFINAL SUMMARY\n" + "="*30)
    print(df_summary.to_string(index=False))
    
    pd.DataFrame(dataset_details).to_csv("benchmark_cell_audit.csv", index=False)
    print("\nDetailed audit saved to 'benchmark_cell_audit.csv'")

if __name__ == "__main__":
    calculate_benchmark_cells()

Starting Standalone EDA: Training Cell Count Analysis...

--- Processing OpenML-CC18 ---
Loaded OpenML-CC18: kr-vs-kp
Loaded OpenML-CC18: letter
Loaded OpenML-CC18: balance-scale
Loaded OpenML-CC18: mfeat-factors
Loaded OpenML-CC18: mfeat-fourier
Loaded OpenML-CC18: breast-w
Loaded OpenML-CC18: mfeat-karhunen
Loaded OpenML-CC18: mfeat-morphological
Loaded OpenML-CC18: mfeat-zernike
Loaded OpenML-CC18: cmc
Loaded OpenML-CC18: optdigits
Loaded OpenML-CC18: credit-approval
Loaded OpenML-CC18: credit-g
Loaded OpenML-CC18: pendigits
Loaded OpenML-CC18: diabetes
Loaded OpenML-CC18: spambase
Loaded OpenML-CC18: splice
Loaded OpenML-CC18: tic-tac-toe
Loaded OpenML-CC18: vehicle
Loaded OpenML-CC18: electricity
Loaded OpenML-CC18: satimage
Loaded OpenML-CC18: eucalyptus
Loaded OpenML-CC18: sick
Loaded OpenML-CC18: vowel
Loaded OpenML-CC18: isolet
Loaded OpenML-CC18: analcatdata_authorship
Loaded OpenML-CC18: analcatdata_dmft
Loaded OpenML-CC18: mnist_784
Loaded OpenML-CC18: pc4
Loaded OpenML-CC1

In [3]:
import pandas as pd
import numpy as np
import gc

def generate_five_number_summary():
    benchmarks = {
        "OpenML-CC18": lambda: local_load_openml_suite(99, "classification", "OpenML-CC18"),
        "OpenML-297": lambda: local_load_openml_suite(297, "regression", "OpenML-297"),
        "AEON-TSC": lambda: local_load_aeon_suite("TSC"),
        "AEON-TSER": lambda: local_load_aeon_suite("TSER")
    }

    all_dataset_sizes = []
    cutoff = 5_000_000

    print("Starting EDA: Training Cell Count Distribution Analysis...\n")

    for benchmark_name, loader_func in benchmarks.items():
        print(f"Loading {benchmark_name}...")
        datasets = loader_func()
        
        for ds in datasets:
            try:
                raw_X = ds["X_train"]
                if hasattr(raw_X, "shape"):
                    shape = raw_X.shape
                    # Standardize logic for 3D vs 2D
                    if len(shape) == 3:
                        rows, cols = shape[0], shape[1] * shape[2]
                    else:
                        rows, cols = shape[0], shape[1] if len(shape) > 1 else 1
                    
                    cells = int(rows * cols)
                    all_dataset_sizes.append({
                        "Benchmark": benchmark_name,
                        "Dataset": ds["name"],
                        "Train_Cells": cells
                    })

                # Clear data immediately
                ds["X_train"] = None
            except Exception:
                pass
        
        gc.collect()

    # Create Analysis DataFrame
    df = pd.DataFrame(all_dataset_sizes)
    
    # 1. Generate the 5-Number Summary
    summary_df = df.groupby('Benchmark')['Train_Cells'].describe(
        percentiles=[.25, .5, .75]
    )[['min', '25%', '50%', '75%', 'max']]

    # 2. Add 'Mean' and 'Count' for extra context
    extra_stats = df.groupby('Benchmark')['Train_Cells'].agg(['count', 'mean'])
    final_report = pd.concat([extra_stats, summary_df], axis=1)

    # Formatting
    pd.options.display.float_format = '{:,.0f}'.format
    
    print("\n" + "=" * 90)
    print("      DISTRIBUTION OF DATASET SIZES (TOTAL TRAINING CELLS)")
    print("=" * 90)
    print(final_report)
    print("-" * 90)

    # 3. Analyze the 5,000,000 Cutoff
    exceed = df[df['Train_Cells'] > cutoff]
    exceed_counts = exceed.groupby('Benchmark').size().rename("Datasets Over Limit")
    exceed_pct = (exceed.groupby('Benchmark').size() / df.groupby('Benchmark').size() * 100).rename("% Over Limit")
    
    impact_report = pd.concat([exceed_counts, exceed_pct], axis=1).fillna(0)
    
    print(f"\nIMPACT ANALYSIS (Limit: {cutoff:,} cells per dataset):")
    print(impact_report.to_string())
    print("=" * 90)

if __name__ == "__main__":
    generate_five_number_summary()

Starting EDA: Training Cell Count Distribution Analysis...

Loading OpenML-CC18...
Loaded OpenML-CC18: kr-vs-kp
Loaded OpenML-CC18: letter
Loaded OpenML-CC18: balance-scale
Loaded OpenML-CC18: mfeat-factors
Loaded OpenML-CC18: mfeat-fourier
Loaded OpenML-CC18: breast-w
Loaded OpenML-CC18: mfeat-karhunen
Loaded OpenML-CC18: mfeat-morphological
Loaded OpenML-CC18: mfeat-zernike
Loaded OpenML-CC18: cmc
Loaded OpenML-CC18: optdigits
Loaded OpenML-CC18: credit-approval
Loaded OpenML-CC18: credit-g
Loaded OpenML-CC18: pendigits
Loaded OpenML-CC18: diabetes
Loaded OpenML-CC18: spambase
Loaded OpenML-CC18: splice
Loaded OpenML-CC18: tic-tac-toe
Loaded OpenML-CC18: vehicle
Loaded OpenML-CC18: electricity
Loaded OpenML-CC18: satimage
Loaded OpenML-CC18: eucalyptus
Loaded OpenML-CC18: sick
Loaded OpenML-CC18: vowel
Loaded OpenML-CC18: isolet
Loaded OpenML-CC18: analcatdata_authorship
Loaded OpenML-CC18: analcatdata_dmft
Loaded OpenML-CC18: mnist_784
Loaded OpenML-CC18: pc4
Loaded OpenML-CC18: pc3